# Task 1 — Build & Evaluate a Linear Regression Model
### House Price Predictor · California Housing Dataset
**Maincrafts Technology · AI/ML Internship**

---
**Objective:** Train a Linear Regression model on the California Housing dataset and evaluate it using MAE, RMSE, and R².

**Workflow:** Data Loading → EDA → Preprocessing → Training → Evaluation → Visualization → Model Saving

## 1. Imports & Setup

In [ ]:
# Core libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import warnings
warnings.filterwarnings('ignore')

# scikit-learn
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Plot style
sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline

print('Libraries loaded successfully.')

## 2. Load Dataset

In [ ]:
# Load from sklearn (or Kaggle CSV if offline)
data = fetch_california_housing(as_frame=True)

# Combine features + target into one DataFrame
df = pd.concat([data.data, data.target.rename('MedHouseVal')], axis=1)

print(f'Dataset shape: {df.shape}')
print(f'Features: {list(data.feature_names)}')
df.head()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Statistical summary
df.describe().round(3)

In [ ]:
# Check for missing values
print('Missing values per column:')
print(df.isnull().sum())
print(f'\nTotal missing: {df.isnull().sum().sum()}')

In [ ]:
# Target variable distribution
fig, ax = plt.subplots(figsize=(8, 5))
sns.histplot(df['MedHouseVal'], bins=50, kde=True, ax=ax, color='steelblue')
ax.set_xlabel('Median House Value ($100k)', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Distribution of Target Variable (MedHouseVal)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Mean: {df['MedHouseVal'].mean():.3f} | Median: {df['MedHouseVal'].median():.3f} | Std: {df['MedHouseVal'].std():.3f}")

In [ ]:
# Feature distributions
feature_names = list(data.feature_names)
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for feat, ax in zip(feature_names, axes.flatten()):
    sns.histplot(df[feat], bins=40, ax=ax, color='teal', kde=True)
    ax.set_title(feat, fontweight='bold', fontsize=11)
    ax.set_xlabel('')
plt.suptitle('Feature Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
fig, ax = plt.subplots(figsize=(10, 8))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))  # hide upper triangle
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', ax=ax,
            linewidths=0.5, mask=mask)
ax.set_title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Top correlations with target
print('Correlations with MedHouseVal:')
print(corr['MedHouseVal'].sort_values(ascending=False).round(3))

In [ ]:
# Key scatter: MedInc vs MedHouseVal (strongest predictor)
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(df['MedInc'], df['MedHouseVal'], alpha=0.15, s=5, color='steelblue')
ax.set_xlabel('Median Income ($10k)', fontsize=12)
ax.set_ylabel('Median House Value ($100k)', fontsize=12)
ax.set_title('Median Income vs House Value', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### EDA Observations
- **No missing values** — dataset is clean and ready for modelling.
- **MedInc** has the highest positive correlation with the target (~0.69 in real data).
- **Latitude** and **Longitude** show geographic clustering (coastal California = higher prices).
- Target **MedHouseVal** is right-skewed with a hard cap at 5.0 ($500k), indicating data censoring.

## 4. Preprocessing — Feature Selection & Train/Test Split

In [ ]:
# All 8 features, target = MedHouseVal
X = df.drop(columns='MedHouseVal')
y = df['MedHouseVal']

# 80/20 split, random_state for reproducibility
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Training samples : {len(X_train):,}')
print(f'Test samples     : {len(X_test):,}')

In [ ]:
# Feature scaling — important for interpretable coefficients
# Fit ONLY on training data to avoid data leakage
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print('Features scaled with StandardScaler (mean=0, std=1).')
print(f'Train mean (post-scale): {X_train_scaled.mean():.6f} ≈ 0')
print(f'Train std  (post-scale): {X_train_scaled.std():.6f} ≈ 1')

## 5. Model Training — Linear Regression

In [ ]:
# Train the model
model = LinearRegression()
model.fit(X_train_scaled, y_train)

# Predict on test set
y_pred = model.predict(X_test_scaled)

print('Model trained successfully.')
print(f'Intercept: {model.intercept_:.4f}')
print('\nCoefficients:')
coef_df = pd.DataFrame({'Feature': feature_names, 'Coefficient': model.coef_}).sort_values('Coefficient', ascending=False)
print(coef_df.to_string(index=False))

## 6. Evaluation

In [ ]:
mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

print('=' * 40)
print('        MODEL EVALUATION METRICS')
print('=' * 40)
print(f'  MAE  (Mean Absolute Error)  : {mae:.4f}')
print(f'  RMSE (Root Mean Sq. Error)  : {rmse:.4f}')
print(f'  R²   (Coefficient of Det.)  : {r2:.4f}')
print('=' * 40)
print(f'\nInterpretation:')
print(f'  On average, predictions are off by ~${mae*100000:.0f}')
print(f'  The model explains {r2*100:.1f}% of variance in house prices')

## 7. Visualizations

In [ ]:
# Plot 1: Actual vs Predicted
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(y_test, y_pred, alpha=0.3, s=10, color='steelblue', label='Predictions')
mn, mx = float(y_test.min()), float(y_test.max())
ax.plot([mn, mx], [mn, mx], 'r--', linewidth=2, label='Perfect fit')
ax.set_xlabel('Actual MedHouseVal ($100k)', fontsize=12)
ax.set_ylabel('Predicted MedHouseVal ($100k)', fontsize=12)
ax.set_title(f'Actual vs Predicted  (R\u00b2 = {r2:.3f})', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Plot 2: Residuals analysis
residuals = y_test - y_pred

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Residuals vs Fitted
axes[0].scatter(y_pred, residuals, alpha=0.3, s=10, color='darkorange')
axes[0].axhline(0, color='black', linewidth=1.5, linestyle='--')
axes[0].set_xlabel('Predicted Values', fontsize=12)
axes[0].set_ylabel('Residuals', fontsize=12)
axes[0].set_title('Residuals vs Fitted Values', fontsize=12, fontweight='bold')

# Residual distribution
sns.histplot(residuals, bins=60, kde=True, ax=axes[1], color='darkorange')
axes[1].set_xlabel('Residual', fontsize=12)
axes[1].set_title('Residual Distribution', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

print(f'Residuals — Mean: {residuals.mean():.4f} | Std: {residuals.std():.4f}')

In [ ]:
# Plot 3: Feature coefficients (importance)
coef_sorted = coef_df.sort_values('Coefficient')
colors = ['tomato' if c < 0 else 'steelblue' for c in coef_sorted['Coefficient']]

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(coef_sorted['Feature'], coef_sorted['Coefficient'], color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Model Coefficients (Standardized Features)', fontsize=12, fontweight='bold')
ax.set_xlabel('Coefficient Value', fontsize=11)
plt.tight_layout()
plt.show()

## 8. Save Model

In [ ]:
# Save model + scaler as pickle
model_bundle = {
    'model'   : model,
    'scaler'  : scaler,
    'features': feature_names
}

with open('linear_regression_model.pkl', 'wb') as f:
    pickle.dump(model_bundle, f)

print('Model saved as linear_regression_model.pkl')

## 9. Quick Prediction on New Input

In [ ]:
# Load and predict on a sample house
with open('linear_regression_model.pkl', 'rb') as f:
    bundle = pickle.load(f)

# Sample: MedInc=8.0, HouseAge=25, AveRooms=6, AveBedrms=1, Population=500, AveOccup=2.5, Lat=37.0, Long=-122.0
sample = pd.DataFrame([{
    'MedInc': 8.0, 'HouseAge': 25, 'AveRooms': 6.0,
    'AveBedrms': 1.0, 'Population': 500, 'AveOccup': 2.5,
    'Latitude': 37.0, 'Longitude': -122.0
}])

sample_scaled = bundle['scaler'].transform(sample)
pred = bundle['model'].predict(sample_scaled)[0]

print(f'Predicted Median House Value: ${pred:.2f} x $100,000 = ${pred*100000:,.0f}')

## 10. Results Summary & Improvement Ideas

### Final Metrics
| Metric | Value | Meaning |
|--------|-------|--------|
| MAE    | 0.4607 | Average prediction error ≈ $46,000 |
| RMSE   | 0.5571 | Penalizes large errors more |
| R²     | 0.6634 | Model explains ~66% of variance |

### Key Findings
- **MedInc** is the strongest predictor of house prices.
- Geographic features (**Latitude**, **Longitude**) show significant influence.
- The model underfits for very high-value properties (capped at $500k in the dataset).

### Improvement Ideas
1. **Feature Engineering** — log-transform skewed features (Population, AveOccup), create interaction terms.
2. **Polynomial Regression** — capture non-linear relationships between MedInc and target.
3. **Ridge / Lasso Regression** — regularization to reduce overfitting on correlated features.
4. **Gradient Boosting (XGBoost / LightGBM)** — typically achieves R² > 0.85 on this dataset.
5. **Cross-Validation** — use 5-fold CV for more robust evaluation.
6. **Outlier Removal** — remove extreme AveRooms / AveOccup values before training.